# Praktische Übung 2: Logistische Regresssion

## Der Titanic Datensatz


Gegeben: Verschiedene Attribute zu den Passagieren auf der Titanic. <br>
Frage: Hat der Passagier das Unglück überlebt?

Da wir den Umgang mit kategorischen Features noch nicht eingeführt haben benutzen wir nur folgende Attribute: <br>
- `Pclass`: Passenger cabin class (1 = 1st, 2 = 2nd, 3 = 3rd)
- `Age`: Age of passenger
- `SibSp`: Number of siblings/spouses aboard
- `Parch`: Number of children/parents aboard
- `Fare`: Passender Fare (brithish pound)
- `Survived`: Did passenger survive? (0 = no, 1 = yes)

## Aufgabe 1

1. Laden Sie den Datensatz aus `titanic.csv` in einen Pandas DataFrame. Die Daten befinden sich im `/data` folder auf GitHub.
2. Erstellen Sie einen neuen DataFrame, der nur die folgenden Spalten enthält: "Survived", "Pclass", "Age", "Fare", "SibSp", "Parch"
3. Entfernen Sie die leeren Felder aus dem DataFrame (d.h. Felder mit `NaN`-Werten) indem sie die Methode `dropna()` auf dem DataFrame aufrufen. <br>Zählen Sie wie viele Zeilen der DataFrame vor und nach dem Aufruf hat. 

In [23]:
# 1.
import pandas as pd
data = pd.read_csv("data/titanic.csv")
# 2.
df = data[["Survived", "Pclass", "Age", "Fare", "SibSp", "Parch"]]
# 3.
print(df.shape)
df = df.dropna()
print(df.shape)
df.head()

(891, 6)
(714, 6)


,Survived,Pclass,Age,Fare,SibSp,Parch
0,0,3,22.0,7.2500,1,0
1,1,1,38.0,71.2833,1,0
2,1,3,26.0,7.9250,0,0
3,1,1,35.0,53.1000,1,0
4,0,3,35.0,8.0500,0,0


## Aufgabe 2

1. Unterteilen Sie den aus Aufgabe 1 entstandenen DataFrame mit Hilfe der Methode `train_test_split` in Trainings- und Testdaten. Nutzen Sie dabei zur besseren Vergleichbarkeit das Argument `random_state=0`.
2. Trainieren Sie eine logistische Regression auf den Trainingsdaten mit "Survived" als Label. 
3. Machen Sie mit dem trainierten Model Vorhersagen auf den Testdaten und berechnen Sie:
    - Accuracy (`~0.69`)
    - Precision (`~0.72`)
    - Recall (`~0.53`)

In [24]:
from sklearn.model_selection import train_test_split
df_no_label = df.drop("Survived", axis=1)
X_train, X_test, y_train, y_test = train_test_split(df_no_label, df["Survived"], test_size=0.2, random_state=0)

from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))

Accuracy: 0.6993006993006993
Precision: 0.723404255319149
Recall: 0.53125


## Aufgabe 3

1. Fügen Sie nun das Geschlecht als weiteres Feature hinzu und führen Sie die Schritte aus Aufgabe 1 und 2 erneut aus (Hinweis: Schauen Sie im Notebook `0_Pandas_Intro` unter "Adding new columns" nach wie das geht). Beachten Sie dabei, dass Geschlecht kein numerischer Wert ist, d.h. Sie müssen daraus ein numerisches Feature erstellen. Beispiel: Feature "isFemale" hat den Wert 1  wenn Sex == "female", sonst 0. Hinweis: Python wandelt Booleans automatisch in Nummern um.
2. In wie weit verbessert sich die Accuracy durch das neue Feature?
3. Versuchen Sie nun auf ähnliche Weise die Spalte "Embarked" als Feature zu nutzen.

In [25]:
#df["isMale"] = [data.loc[idx, "Sex"] == "male" for idx in df.index]

df = data[["Survived", "Pclass", "Age", "Fare", "SibSp", "Parch", "Sex"]]
df = df.dropna()
df["isMale"] = df["Sex"] == "male"
df = df.drop("Sex", axis=1)

df_no_label = df.drop("Survived", axis=1)
X_train, X_test, y_train, y_test = train_test_split(df_no_label, df["Survived"], test_size=0.2, random_state=0)

model = LogisticRegression()
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))

Accuracy: 0.8461538461538461
Precision: 0.8387096774193549
Recall: 0.8125


In [26]:
df = data[["Survived", "Pclass", "Age", "Fare", "SibSp", "Parch", "Sex", "Embarked"]]
df = df.dropna()
df["isMale"] = df["Sex"] == "male"

df["Embarked_C"] = df["Embarked"] == "C"
df["Embarked_Q"] = df["Embarked"] == "Q"
df["Embarked_S"] = df["Embarked"] == "S"

df.drop(["Sex", "Embarked"], axis=1, inplace=True)

df_no_label = df.drop("Survived", axis=1)
X_train, X_test, y_train, y_test = train_test_split(df_no_label, df["Survived"], test_size=0.2, random_state=0)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions))
print("Recall:", recall_score(y_test, predictions))


Accuracy: 0.8181818181818182
Precision: 0.8431372549019608
Recall: 0.7049180327868853


## Bonus

Welches sind die wichtigsten und unwichtigsten Features des Modells fur die Vorhersage? 

In [27]:
feature_importance = pd.Series(model.coef_[0], index=X_train.columns)
feature_importance = feature_importance.reindex(feature_importance.abs().sort_values(ascending=False).index)

feature_importance

isMale       -2.416225
Pclass       -1.065517
SibSp        -0.414475
Embarked_C    0.398015
Embarked_Q   -0.278373
Embarked_S   -0.119455
Age          -0.039534
Parch        -0.033735
Fare          0.002381
dtype: float64

In [28]:
featureWeights = model.coef_[0]
print(featureWeights)
featureNames = X_train.columns
print(featureNames)

[-1.06551659e+00 -3.95342421e-02  2.38127122e-03 -4.14474800e-01
 -3.37354582e-02 -2.41622474e+00  3.98014970e-01 -2.78372676e-01
 -1.19455371e-01]
Index(['Pclass', 'Age', 'Fare', 'SibSp', 'Parch', 'isMale', 'Embarked_C',
       'Embarked_Q', 'Embarked_S'],
      dtype='str')


In [30]:
df = pd.DataFrame(data={'feature': featureNames, 'importance': featureWeights})
df["importanceAbs"] = df["importance"].abs()
df.sort_values("importanceAbs", ascending=False)

,feature,importance,importanceAbs
5,isMale,-2.416225,2.416225
0,Pclass,-1.065517,1.065517
3,SibSp,-0.414475,0.414475
6,Embarked_C,0.398015,0.398015
7,Embarked_Q,-0.278373,0.278373
8,Embarked_S,-0.119455,0.119455
1,Age,-0.039534,0.039534
4,Parch,-0.033735,0.033735
2,Fare,0.002381,0.002381
